# Orbit Wars — RL Modeli ve Eğitim Notları

Bu notebook, repoda güncel olan **GroupedPlanet Set Transformer + Beta head + PPO + delayed reward**
yığınının teorik temellerini ve pratik ayrıntılarını toplu halde açıklar. Odak, kod gezisi değil;
**neden böyle tasarlandığının matematiksel ve RL teorisindeki karşılığıdır.**

İlgili dosyalar:

- `model.py` — full Set Transformer policy/value ağı; Beta head yardımcıları (`_beta_concentrations`, `_make_ship_dist`).
- `model_grouped_planet.py` — **default variant**: planet token'lar üzerinde transformer + per-source/per-pair head'ler + pair-conditioned Beta ship head.
- `obs_builder.py` — gözlem vektörü ve edge bias üretimi; `decode_action_with_launch` ile env aksiyon decode'u.
- `rl/ppo.py` — clipped surrogate PPO güncellemesi; KL early-stop; Beta α/β + ratio quantile logging.
- `rl/buffer.py` — on-policy rollout buffer, GAE, `add_delayed_reward`.
- `rl/rewards.py` — reward konfigürasyonu ve step shaping.
- `rl/rollout.py` — self-play episode toplayıcı, delayed reward dağıtımı.
- `rl/production_reward_tracker.py`, `rl/fleet_tracker.py`, `rl/defense_tracker.py`, `rl/planet_loss_tracker.py` — delayed credit assignment mekanizmaları.

Notebook çalıştırılabilir değildir; amacı kuramsal bir referans olmaktır. Bazı kısa kod blokları,
kod tabanına bir-bire-bir bağlantı kurmak için eklenmiştir.

> **Önemli güncelleme (2026-05):** Eski 5-bin discrete `ship_logits` tasarımı yerine **continuous
> Beta head** kullanılıyor. Eski `model_planet_only.py` referansları geçersizdir; default model
> `model_grouped_planet.GroupedPlanetOrbitWarsPolicy`'dir. Beta α, β concentrations [1, 30] aralığına
> tanh ile soft-cap'lenmiştir (delta-collapse + PPO ratio explosion önlemi).

## 1. Problem: Self-Play Zero-Sum MDP

Orbit Wars, iki (veya 4) oyunculu, ayrık zamanlı, kısmen gözlemlenebilir ama her oyuncuya simetrik bilgi veren bir strateji oyunudur. Eğitim açısından onu aşağıdaki gibi formalize ediyoruz:

- Durum: $s_t$ — environment'ın tam durumu (gezegenler, filolar, üretim, komet vs).
- Oyuncu $i$ için gözlem: $o_t^{(i)} = \phi_i(s_t)$ — kendi perspektifine göre normalize edilmiş feature tensörleri (`obs_builder.build_observation`).
- Oyuncu $i$ için aksiyon: $a_t^{(i)} = (\text{src}, \text{tgt}, \text{ship\_value}, \text{is\_noop})$, **ship_value ∈ [0, 1] continuous Beta sample**.
- Geçiş: $s_{t+1} = T(s_t, a_t^{(0)}, a_t^{(1)}, \dots)$ — tüm oyuncu aksiyonları eşzamanlı uygulanır. Her oyuncu turn başına çoklu launch verebilir (rollout cap = 8).
- Ödül: $r_t^{(i)}$ — terminal galibiyet/mağlubiyet + dense shaping + delayed credit.

**Self-play**: tek bir politika $\pi_\theta$ her iki tarafta da oynatılır. Bu, **non-stationary ama simetrik** bir ortama sebep olur; çünkü $\pi$ güncellendikçe rakibin davranışı da değişir. PPO'nun on-policy özelliği ve küçük güncelleme adımı (clip + KL early-stop), bu kaymayı tolere etmek için tasarımın tutarlı kısmıdır.

Amaç: Oyuncunun kendi ödül toplamının beklentisini maksimize etmek.

$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T} \gamma^{\,t}\, r_t \right].
$$

Zero-sum yapısı nedeniyle bir oyuncunun değeri, diğerinin değerinin yaklaşık olarak negatifidir; terminal ödülde $+1/-1$ simetrisi ile bu açıkça korunur (`normalize_terminal_rewards`).

## 2. Gözlem (Observation) Tasarımı

`obs_builder.py` env durumunu sabit şekilli numpy tensörlerine çevirir. Bu tensörler Set Transformer'in girdisidir. Temel ilkeler:

1. **Permutation invariance**: gezegen ve filo slotları set olarak sunulur, maske ile padding ayırt edilir.
2. **Kutupsal simetri**: oyunun merkezinde güneş var; özellikler $(r,\theta)$ tabanlı çıkarılır, $(\cos\theta, \sin\theta)$ ile döngüsellik korunur.
3. **Geometrik ön bilgi**: modelin uzaklık/açı öğrenmesi için **edge bias** tensörü hazırlanır ve attention'a eklenir.
4. **Aksiyon maskeleme**: yalnızca yeterli gemisi olan kendi gezegenlerimiz kaynak olabilir (`source_mask`).

### 2.1 Planet features (`PLANET_FEATURE_DIM = 34`)

Her gezegen için 34 boyutlu vektör üretilir (indeksler `obs_builder` içindeki `PF_IDX_*` sabitleriyle eşleşir):

- **Sahiplik (5)**: `is_self`, `enemy_slot_0/1/2` (4-player için 3 düşman one-hot), `is_neutral`.
- **Ekonomi (4)**: `ships/100`, `production/5`, `radius/5`, `log_ships`.
- **Konum (4)**: `r_sun/50`, `cos(θ)`, `sin(θ)`, `omega` (rotating ise).
- **Tip (3)**: `is_static`, `is_rotating`, `is_comet`.
- **Self gelen filolar (4)**: ETA bucket'lar `[0,5], (5,15], (15,30], 30+`.
- **Düşman gelen filolar (4)**: aynı 4 ETA bucket'ı.
- **Tehdit özeti (10)**: self/enemy total, min ETA, soonest/largest enemy ships+ETA, `net_threat`, `capture_value`, `defense_deficit`.

Bu zenginleştirme sayesinde grouped_planet model, fleet token'ları olmadan da savunma/saldırı zamanlamasını kavrayabilir.

### 2.2 Fleet features

`f_max` slotluk filo tensörü üretilir (`FLEET_FEATURE_DIM = 11`). Grouped_planet variant'ı bu tensörü
**ENCODE etmiyor**: tüm fleet bilgisi planet feature içindeki incoming bucket aggregate'lerine
sıkıştırılıyor. Saved batch'lerde alan kalır ama model'in `_build_tokens`'ı sadece `planet_feats` kullanır.

### 2.3 Global features

`GLOBAL_FEATURE_DIM = 14` boyutlu vektör: oyun turu, sahip olduğum/rakibin gezegen sayısı, toplam gemi
oranı, üretim oranları, pozisyonel özet ve harcanmış süre gibi CLS-token benzeri global özetler içerir.
Grouped_planet'te attention pool seed'leri ile beraber `global_summary`'ye giriyor.

### 2.4 Edge bias

Her iki token arasında **6 boyutlu** bir edge özelliği hesaplanır (`EDGE_FEATURE_DIM = 6`):

$$
e_{ij} = \big(d_{ij}/d_{\max},\; \Delta x,\; \Delta y,\; \sin\Delta\varphi_{\text{sun}},\; \cos\Delta\varphi_{\text{sun}},\; \mathbb{1}[\text{crosses sun}]\big).
$$

Aynı yörünge hızına sahip iki gezegen için $\Delta\varphi_{\text{sun}}$ **zaman-değişmez** bir
büyüklüktür; bu da modelin yörünge ilişkilerini öğrenmesini kolaylaştırır. Son kanal `crosses_sun`,
düz çizgi rotasının güneş yarıçapından geçip geçmediğini söyler — navigation seviyesinde reddedilen
rotaları model'e önsel olarak bildirir.

Edge bias, **head başına additive attention bias** olarak projekte edilir:

$$
\text{scores} = \frac{QK^\top}{\sqrt{d}} + B_{\text{edge}}\;, \quad B_{\text{edge}} \in \mathbb{R}^{H\times P\times P}.
$$

Bu, modelin koordinatlardan mesafe yeniden öğrenmesini beklemek yerine geometriyi doğrudan önsel
bilgi olarak vermek demek. Grouped_planet variant'ında edge bias **sadece planet-planet alt blok**
için kullanılır (fleet token yok).

## 3. Model: Grouped Planet Set Transformer Policy / Value

`GroupedPlanetOrbitWarsPolicy` (`model_grouped_planet.py`) **default variant**'tır.
`OrbitWarsPolicy` (full, `model.py`) legacy yoldur ve fleet token'larını ayrı encode eder.

### Token düzeni

- **Grouped (default)**: sadece `[planet_0, ..., planet_{P-1}]` token'ları (T = P_max). Fleet bilgisi
  planet feature içindeki incoming bucket aggregate'lerinde; global bilgi pool seed'leriyle
  cross-attention üzerinden çekiliyor.
- **Full (legacy)**: `[planet_0, ..., planet_{P-1}, fleet_0, ..., fleet_{F-1}, GLOBAL]`,
  $T = P_{\max} + F_{\max} + 1$.

### 3.1 Biased multi-head attention (full variant)

Standart çok kafalı dikkat mekanizmasına ek olarak, her pozisyon çifti için additive bias:

$$
\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + B_{\text{edge}} + M_{\text{pad}}\right) V,
$$

- $B_{\text{edge}}$: yukarıdaki geometrik bias.
- $M_{\text{pad}}$: padded key'ler için $-\infty$ maskesi.

Padded satırlarda softmax NaN üretir; `torch.nan_to_num` ile güvenli sıfırla doldurulur.

### 3.2 Pre-norm Transformer blokları

$$
x \leftarrow x + \text{Attn}(\text{LN}(x))\;,\quad x \leftarrow x + \text{FFN}(\text{LN}(x)).
$$

Varsayılan: `d_model=128`, `n_heads=4`, `n_layers=3-4`, FFN genişleme çarpanı 4 ve GELU aktivasyon.

### 3.3 Grouped variant'ta induced attention seçeneği

Grouped_planet `attention_type='induced'` modunu da destekler: ISAB ile O(P·M) maliyetli attention,
edge bias yok. CPU smoke test'te tercih edilebilir.

### 3.4 Policy head'leri ve action semantiği

Modelin `forward` çıktısı (PolicyOutput dataclass):

- **`src_logits`**: `(B, P_max + 1)`. İlk P slot per-source `noop_raw[s]` (kategorik içinde NOOP-vs-launch); son slot global NOOP (per-source noop_raw'ın max'ı).
- **`tgt_logits`**: `(B, P_max, P_max)`. Bilinear pair pointer + `source_launch[s]` shift:
  $\text{tgt\_logits}[s, t] = (q_s \cdot k_t)/\sqrt{d} + \ell_s$.
- **`ship_params`**: `(B, P_max, P_max, 2)`. **Pair-conditioned Beta head**: her `(src, tgt)` için raw `(α, β)` logitleri. `_beta_concentrations`: $\alpha,\beta = 1 + (\text{MAX}-1)\tanh(\text{softplus}(x)/(\text{MAX}-1))$, MAX=30.
- **`value`**: `(B,)`. Global pool seed'lerinin fuze edilmiş özetinden critic.
- **`aux_owner_logits`**: `(B, P_max, 5)` — yardımcı self-supervision (her gezegenin owner'ı).

### Rollout aksiyon semantiği — `act_source(...)`

Training rollout `policy.act(...)` (joint argmax) değil, **source-by-source döngüde `act_source(...)`** kullanır:

```text
for src_slot in current_source_slots:
    action_logits = [ noop_raw[src],
                      qk[src, t_0] + ℓ_src,
                      qk[src, t_1] + ℓ_src,
                      ...,
                      qk[src, t_{P-1}] + ℓ_src ]    # length P+1
    action_idx ~ Categorical(softmax(action_logits))
    if action_idx == 0:        # NOOP
        is_noop = True
    else:
        tgt = action_idx - 1
        ship_value ~ Beta(α[src, tgt], β[src, tgt])    # ∈ (0, 1)
```

Buradaki kritik detay: **NOOP, target'larla aynı softmax içinde yarışır** — modelin "src'den t_3'e
gitmek mantıklı ama t_5 zayıf, t_5 yerine NOOP yap" tercihi tek normalleşme havuzunda ifade edilir.

Tek source kararının log-prob'u:

$$
\log \pi(a_i|s_i, \text{src}_i) = \log p(\text{action\_idx}_i \mid s_i, \text{src}_i)
+ \mathbb{1}[\neg\text{noop}_i] \log p_{\text{Beta}}(\text{ship\_value}_i \mid s_i, \text{src}_i, \text{tgt}_i).
$$

PPO tarafında aynı semantik `evaluate_source_actions(...)` ile tekrar hesaplanır. Stored batch
içindeki `src`, `tgt`, `ship_value`, `is_noop`, `old_log_prob` alanları bu source-conditioned
faktörizasyona göredir.

### 3.5 Beta head soft-cap (kritik)

`_beta_concentrations` α ve β'yı **[1, 30] aralığına** sıkıştırır:

```python
α = 1 + 29 · tanh(softplus(raw_α) / 29)
β = 1 + 29 · tanh(softplus(raw_β) / 29)
```

Cap olmadan α, β → ∞'a giderek Beta'yı delta function'a çökertiyordu. Bu durumda log_prob
$\log p_{\text{Beta}}(\sigma; \alpha, \beta)$ keskinleşip PPO importance ratio
$\exp(\log p_{\text{new}} - \log p_{\text{old}})$ kontrolden çıkıyor (gözlenen: KL>100,
clip_fraction>0.9, polL spike ≫ 1000). Cap ile log-prob limitli, ratio temkinli.

### 3.6 Angle nasıl hesaplanıyor?

Policy **açı üretmez**. `obs_builder.decode_action_with_launch(...)` policy çıktısını env formatına çevirir:
`navigation.compute_launch(src, tgt, ships, planets, angular_velocity)` hareketli hedef için intercept
ETA hesaplar, açıyı türetir, güneş ve diğer gezegen çarpışma kontrollerinden geçmeyen rotaları reddeder.
Output: `[from_planet_id, angle_in_radians, ships]`.

### 3.7 İki model variant'ı

- **Grouped (default, `model_grouped_planet.py`)**: P-token attention; pair-conditioned ship Beta head.
- **Full (legacy, `model.py`)**: planet + fleet + global token; per-source ship Beta head (P×2).

Model seçimi `train.py --model-variant {grouped_planet, full}` ile yapılır.

### 3.8 Güncel mimari snapshot ve bilinen sınırlar

#### Encoder

Default eğitim yolu `GroupedPlanetOrbitWarsPolicy` kullanır:

```text
planet_feats: (B, P_max=44, 34)
edge_bias   : (B, P_max + F_max + 1, P_max + F_max + 1, 6)
              # ama grouped variant sadece [:P, :P] alt bloğunu okur
T (encoder) : P_max
```

Tek tek fleet token yoktur. Filo bilgisi planet feature içindeki incoming ETA bucket aggregate'larına
sıkıştırılır. Bu, token sayısını küçültür ve PPO training'i daha stabil yapar; karşılığında tek tek
fleet trajectory bilgisini modele doğrudan vermez.

#### Actor head'leri

Güncel actor şu yapıdadır:

```text
src_head[s]    = Linear([planet_tok[s], global_summary])  → ℓ_s   (per-source launch propensity)
noop_head[s]   = Linear([planet_tok[s], global_summary])  → noop_raw[s]
tgt_logits[s,t] = (q_s · k_t)/√d_model + ℓ_s              # bilinear pointer
ship_params[s,t] = MLP([planet_tok[s], planet_tok[t], global_summary])  → (α, β)
                   sonra _beta_concentrations ile [1, 30]'a soft-cap
```

**Önemli güncelleme:** `ship_head` artık **target-conditioned**. Her `(src, tgt)` pair için ayrı
`(α, β)` üretilir. Bu, "agent doğru target'a saldırır ama yanlış gemi miktarı seçer" sorununu
mimari olarak çözmek için yapılan bir değişiklikti.

#### Rotation bilgisi modelde var mı?

Evet. Planet feature içinde rotating bilgisi açıkça vardır:

```text
is_static, is_rotating, is_comet, omega, r_sun, cos(theta), sin(theta)
```

Ayrıca edge bias içinde güneş merkezli faz farkı bulunur:

```text
sin(phi_j - phi_i), cos(phi_j - phi_i), crosses_sun
```

Bu yüzden model bir target'ın rotating olduğunu ve source-target orbital ilişkisini görebilir.
Ancak şu bilgi **explicit** değildir:

```text
bu source-target çifti için şu an launch valid mi?
ETA kaç?
kaç turn sonra açı/fırsat kapanacak?
```

Bu bilgiler şu anda policy input'una feature olarak verilmez; launch feasibility
`decode_action_with_launch -> navigation.compute_launch` aşamasında deterministik olarak kontrol
edilir. Invalid trajectory varsa action env'e gönderilmez ve reward/invalid-action mekanizması
üzerinden öğrenilir.

#### Macro-agent ilkesi

Policy angle üretmez ve üretmemelidir. Aksiyon uzayı macro düzeydedir:

```text
hangi kaynaktan?
hangi hedefe?
ne kadar gemiyle? (Beta sample × SHIP_REQUEST_CAP)
launch mı no-op mu?
```

Açı, intercept ETA, spawn offset, sun collision ve planet sweep collision `navigation.py` tarafından
deterministik çözülür.

#### Kısa mimari risk listesi

- `ship_head` artık pair-conditioned ✓ (önceki risk kalktı).
- **Beta head soft-cap [1, 30]** — concentration patlaması artık sınırlı ✓.
- Future opportunity/reachability feature yok; rotating gezegenlerde fırsat penceresi tamamen
  learned representation'a kalır.
- Grouped_planet model tekil fleet tokenlarını görmez; incoming aggregate bucket'ları yeterli olmadığı
  durumlarda hassas savunma/saldırı zamanlaması zorlaşabilir.
- Reward attribution doğru olsa bile source başına aynı turn içinde çok sayıda transition üretilir;
  global step reward bu transition'lara paylaştırılır.

In [1]:
# Mimari hızlı özet (çalıştırmak için opsiyonel)
import os, sys, pathlib
repo_root = pathlib.Path().resolve()
if repo_root.name == 'docs':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

from model_grouped_planet import GroupedPlanetOrbitWarsPolicy
from obs_builder import PLANET_FEATURE_DIM, GLOBAL_FEATURE_DIM, EDGE_FEATURE_DIM, P_MAX_DEFAULT

policy = GroupedPlanetOrbitWarsPolicy(
    p_max=P_MAX_DEFAULT,
    planet_dim=PLANET_FEATURE_DIM,
    global_dim=GLOBAL_FEATURE_DIM,
    edge_dim=EDGE_FEATURE_DIM,
    d_model=128, n_heads=4, n_layers=3,
)
total_params = sum(p.numel() for p in policy.parameters())
print(f"Variant       : grouped_planet (default)")
print(f"P_MAX         : {P_MAX_DEFAULT}")
print(f"planet_dim    : {PLANET_FEATURE_DIM}")
print(f"global_dim    : {GLOBAL_FEATURE_DIM}")
print(f"edge_dim      : {EDGE_FEATURE_DIM}")
print(f"Token count T : {policy.encoder.p_max}")
print(f"Parametre     : {total_params:,}")


Variant       : grouped_planet (default)
P_MAX         : 44
planet_dim    : 34
global_dim    : 14
edge_dim      : 6
Token count T : 44
Parametre     : 950,310


## 4. Öğrenme Algoritması: PPO + GAE

Politika güncellemesi **Proximal Policy Optimization (clip variant)** ile yapılır (`rl/ppo.py`).

### 4.1 Clipped surrogate objective

$r_t(\theta) = \pi_\theta(a_t|s_t) / \pi_{\theta_{\text{old}}}(a_t|s_t)$ olmak üzere:

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[\min\!\left(r_t\hat A_t,\; \text{clip}(r_t, 1-\epsilon, 1+\epsilon) \hat A_t\right)\right].
$$

Bu terim, politika dağılımının **bir adımda çok fazla kaymasını** yapısal olarak sınırlar.

### 4.2 PPO batch içinde hangi action log-prob tekrar hesaplanıyor?

Rollout `act_source(...)` kullandığı için PPO update **`evaluate_source_actions(...)`** çağırır:

```text
out = policy(batch observation)
ev = policy.evaluate_source_actions(
    out,
    src=batch["src"],
    tgt=batch["tgt"],
    ship_value=batch["ship_value"],   # float ∈ [0, 1] (Beta sample)
    is_noop=batch["is_noop"],
)
new_logp = ev["log_prob"]
ratio = exp(new_logp - old_log_prob)
```

Bu kontrol kritik: stored `old_log_prob` ile update sırasındaki `new_logp` aynı source-conditioned faktörizasyonu kullanmalıdır. Aksi halde PPO ratio yanlış olur.

### 4.3 Value loss (opsiyonel clipping)

$$
L^{VF}(\theta) = \frac{1}{2}\max\left((V_\theta - R)^2,\; (V^{\text{clip}} - R)^2\right),
$$

burada $V^{\text{clip}} = V_{\theta_{\text{old}}} + \text{clip}(V_\theta - V_{\theta_{\text{old}}}, -\epsilon_V, \epsilon_V)$.

### 4.4 Entropi bonusu

Toplam kayıp:

$$
L(\theta) = L^{\text{CLIP}} + c_1 L^{VF} - c_2\, \mathbb{E}\!\left[ H[\pi_\theta](s) \right].
$$

Per-source entropi:

$$
H[\pi(\cdot \mid s, \text{src})] = H_{\text{action}} + \Pr[\text{launch} \mid s, \text{src}] \cdot H_{\text{Beta}}(\alpha, \beta).
$$

**Önemli:** `Pr[launch]` mevcut policy'nin marjinal launch olasılığıdır (`1 - softmax(action_logits)[..., 0]`),
**realize edilmiş `valid` flag'i değil**. Eski formülasyon (realize flag) PPO epoch'ları içinde launch
oranı kayarken entropy gradient'ini bias'lardı. Yeni formülasyon population-level entropi vererek
bu sorunu çözer (`model.py:507-516`, `model_grouped_planet.py:534-544`).

### 4.5 Generalized Advantage Estimation

GAE-$\lambda$ yüksek varyanslı Monte Carlo getirisi ile düşük varyanslı TD(0) arasında interpolasyon sağlar:

$$
\delta_t = r_t + \gamma V(s_{t+1})\,\mathbb{1}[\neg\text{done}_t] - V(s_t),\qquad
\hat A_t = \sum_{l \geq 0} (\gamma\lambda)^l \delta_{t+l}.
$$

`rl/buffer.py` içinde her oyuncunun transition'ları için ayrı rekürsif hesaplama yapılır (`compute_gae`). Default: `gamma=0.99`, `lam=0.95`.

### 4.6 Advantage normalizasyonu

`iter_minibatches(...)` içinde advantage'lar tüm rollout buffer üzerinde ortalama=0, std=1'e normalize edilir. Saved game batch dosyalarında ise `batch_from_indices(..., normalize_advantages=False)` kullanıldığı için ham GAE advantage saklanır.

### 4.7 KL early stopping (yeni)

`PPOConfig.target_kl: Optional[float] = 0.05` parametresi eklendi. Her PPO inner-epoch sonunda
o epoch'un mean approx_kl'i kontrol edilir; `> target_kl` ise kalan epoch'lar atlanır:

```text
for ppo_epoch in range(K):
    for minibatch in buffer:
        ...PPO update...
        epoch_kls.append(approx_kl)
    if mean(epoch_kls) > target_kl:
        break  # rollout snapshot'ından çok uzaklaştık, fresh data topla
```

Bu, gözlenen "KL=170, clip=0.96, polL=+5977" patolojisini önler. CLI'da `--target-kl 0.05`
(default 0.05; negatif veya 0 ⇒ disabled).

### 4.8 Diagnostic logging

`PPOStats` dataclass'ına eklenen yeni alanlar:

| Alan | Anlam |
|---|---|
| `ratio_q05/q50/q95` | Importance ratio quantile'ları (sağlıklı: 0.8-1.2 arası dar bant) |
| `ship_alpha_mean / ship_beta_mean` | Beta concentration ortalamaları (chosen pair, non-noop) |
| `epochs_run` | KL early-stop nedeniyle kaç epoch tamamlandı |

Console satırı: `r=[q05,q50,q95] αβ=(α,β) e=run/total`. Bu metrikler trust-region sağlığını
tek bakışta gösterir.

## 5. Ödül (Reward) Sistemi — Güncel Kod Durumu

Bu bölüm model incelemesi için reward sinyalinin PPO batch’e nasıl girdiğini açıklar. Burada tasarım önerisi değil, mevcut kod akışı anlatılır.

### 5.1 Terminal ödül

`normalize_terminal_rewards(raw_rewards, draw_value=...)` terminal env reward’unu normalize eder:

- Tek kazanan varsa: kazanan `+1`, diğerleri `-1`.
- Tek kazanan yoksa: herkes `draw_value` alır.

`production_simple` presetinde güncel `draw_terminal_reward=-1.0` olarak ayarlıdır. Bu değer CLI’dan `--draw-terminal-reward` ile override edilebilir.

### 5.2 Step shaping

`compute_step_reward(...)` şu bileşenleri hesaplayabilir:

- capture/denial production delta,
- ship diff,
- ship loss,
- invalid action,
- step cost,
- terminal reward.

`production_simple` presetinde step-level capture/ship shaping kapalıdır (`capture_value_weight=0`, `denial_value_weight=0`, `ship_diff_weight=0`, `ship_loss_weight=0`). Bu yüzden ana sinyal launch transition’larına delayed/immediate olarak eklenen tracker reward’larından gelir.

### 5.3 Launch transition reward kaynakları

Bir source için transition buffer’a eklendikten sonra reward şu yollardan değişebilir:

```text
immediate:
  invalid action penalty
  airborne cost
  source defense deficit penalty
  terminal reward share if terminal step

delayed:
  fleet combat/capture outcome
  production capture outcome
  defense hold outcome for NO-OP holds
```

`RolloutBuffer.add_reward(idx, value)` aynı transition’ın reward alanını artırır/azaltır. `add_delayed_reward(idx, reward, event_turn=t, discount=d)` ise event gecikmesine göre `reward * d**delay` ekler.

### 5.4 production_simple formülleri

Güncel preset:

```text
reward_scale = 0.001
fleet_airborne_weight = 1.0
fleet_kill_weight = 1.0
fleet_waste_weight = 1.0
planet_loss_weight = 1.0
delayed_reward_discount = 1.0
```

Capture reward:

$$
R_{capture}= production \cdot (T_{max}-t) \cdot opponent\_multiplier \cdot ship\_share \cdot reward\_scale
$$

Fleet combat reward:

$$
R_{combat}= \frac{enemy\_ships\_killed}{my\_total\_ships\_at\_launch}\cdot reward\_scale
$$

Combatta hedefe ulaşan ama capture edemeyen filolar için `own_waste` cezası kaldırılmıştır; sadece `destroyed_enroute` durumunda waste negatif yazılır.

Airborne cost:

$$
R_{airborne}= - fleet\_airborne\_weight \cdot ships\_sent \cdot ETA \cdot risk\_multiplier \cdot reward\_scale
$$

Source defense deficit penalty:

$$
R_{deficit}= -\max(0, enemy\_incoming - ships\_remaining\_after\_launch) \cdot planet\_loss\_weight \cdot reward\_scale
$$

### 5.5 Son diagnostic bulgusu

Son incelemede reward/PPO kodu değiştirilmeden şu ölçüm yapıldı:

- Checkpoint: `production_simple_v7_softening/policy_iter00009.pt`
- Aynı rollout batch üzerinde launch vs no-op reward/advantage karşılaştırıldı.
- PPO `evaluate_source_actions(...)` ile stored `old_log_prob` semantiği uyumlu şekilde tekrar hesaplandı.

Ölçüm sonucu:

```text
base:
  reward launch mean ≈ -0.433
  reward noop mean   ≈ +0.000
  normalized advantage launch mean ≈ -0.168
  normalized advantage noop mean   ≈ +0.063

fleet_airborne_weight=0 ablation:
  reward launch mean ≈ +0.027
  reward noop mean   ≈ +0.000
  normalized advantage launch mean ≈ +0.065
  normalized advantage noop mean   ≈ -0.024
```

Aynı batch üzerinde tek PPO update denemesi:

```text
before mean p_launch ≈ 0.241
after  mean p_launch ≈ 0.202
delta launch logit mean ≈ -0.259
```

Bu, mevcut batch etiketlerinin launch aksiyonlarını ortalamada azaltma yönünde gradient ürettiğini gösterir. Bu hücre sadece mevcut modeli/batch’i incelemek için kaydedilmiştir; reward tasarım kararını otomatik olarak değiştirmez.

## 6. Rollout ve Training Pipeline

Bir epoch (iteration) içinde şu sıra izlenir (`rl/train.py` + `rl/rollout.py`):

1. **Self-play rollout**: tek bir policy her iki tarafta oynatılır.
   - Her oyuncu için `build_observation` ile planet/edge tensörleri üretilir.
   - `policy.act_source` ile aksiyon örneklenir; log-prob ve value saklanır.
   - Transition buffer’a eklenirken `turn=t` de kaydedilir; böylece gelecekteki delayed ödüller doğru step’e bağlanır.
   - Fleet launch’ları ilgili trackerlara **buffer_idx** ile kayıt edilir.
   - Env.step’ten sonra: fleet arrival, defense hold, production capture/loss, planet loss olaylarından gelen ödüller `add_delayed_reward` ile buffer’a yazılır.
2. **GAE**: episode sonu bootstrap $V=0$ ile `compute_gae` çalıştırılır; her oyuncu ayrı ayrı rekürsif olarak işlenir.
3. **PPO update**: $K$ epoch boyunca shuffled minibatchler üzerinde clipped surrogate loss.
4. **Logging/checkpoint**: KL, clip fraction, value loss, entropi, grad norms ve update ratio metrikleri JSONL’e yazılır, `latest.pt` ve `policy_iter*.pt` kaydedilir.

### 6.1 İki-oyunculu buffer

Her env step’inde iki transition eklenir (p0 ve p1). GAE hesaplanırken `compute_gae` oyuncuları ayırıp her biri için ayrı rekürsyon yapar; böylece iki oyuncunun getirileri birbirine karışmaz.

### 6.2 Seed yönetimi

`seed = train_cfg.seed * 100000 + it * 1000 + ep` formülü, iterasyon ve episode ikilileri için deterministik ama çeşitli seedler üretir; aynı başlangıç konfigürasyonuyla eğitim tekrar üretilebilir.

## 7. Hiperparametreler ve Teşhis Notları

Varsayılan `TrainConfig` ve `PPOConfig` (`rl/train.py`, `rl/ppo.py`):

| Hiperparametre | Default | Sezgi |
|---|---|---|
| `gamma` | 0.99 | Uzun-vadeli plan: ~100 adımlık horizon. |
| `gae_lambda` | 0.95 | Yüksek MC / düşük TD arasında tipik PPO dengesi. |
| `learning_rate` | 3e-4 | Adam ile Transformer için başlangıç değeri. Beta head düşmesi için 5e-5'e düşürülebilir. |
| `clip_range` | 0.2 | Politika güncelleme büyüklüğü sınırı. |
| `value_clip_range` | 0.2 | Critic aşırı güncellemesini engeller. |
| `ppo_epochs` | 4 | Aynı buffer üzerinden 4 tur geçiş. KL early-stop ile dinamik kesilir. |
| `target_kl` | 0.05 | KL early-stop eşiği. Aşılırsa kalan inner epoch'lar atlanır. |
| `batch_size` | 256 | Minibatch boyutu. |
| `entropy_coef` | 0.05 | Exploration bonusu. |
| `max_grad_norm` | 0.5 | Gradient clipping. |
| `episodes_per_iter` | 1 | Her oyundan sonra update. |
| `max_episode_steps` | 500 | Env'in doğal episode sınırı. |
| `model_variant` | `grouped_planet` | Pair-conditioned Beta head + planet-only token. |

### 7.1 KL ve clip fraction teşhisi

- `approx_kl` çok büyükse: target_kl etkin değilse, lr veya clip_range fazla olabilir.
- `clip_fraction` yüksekse: PPO çoğu örneği kesiyor olabilir. >0.4 alarm.
- `entropy` hızla 0'a inerse veya negatife giderse: Beta delta'ya çöküyor olabilir; soft cap [1, 30] aktif mi kontrol et.
- `ratio_q05/q95` geniş açılırsa (örn. q05<0.5, q95>2): trust-region kırılıyor, lr düşür veya target_kl sıkılaştır.
- `ship_alpha/beta` 30'a yapışırsa: Beta hâlâ çok concentrate, entropy_coef artır veya cap'i düşür (örn. 20'ye).
- `value_loss` sürekli artıyorsa veya çok büyük başlıyorsa: reward scale / return scale kontrol edilmeli.

### 7.2 Launch/no-op collapse teşhisi için batch metrikleri

Sadece `launch/p` loguna bakmak yeterli değildir. Bir rollout batch üzerinde şu ayrımlara bakılmalı:

```text
is_noop == False:
  reward mean / return mean / advantage mean
is_noop == True:
  reward mean / return mean / advantage mean
```

Eğer normalize advantage ortalaması launch için negatif, no-op için pozitifse PPO update launch logitlerini düşürür. Bu davranış PPO hatası değildir; batch'in policy'ye verdiği etikettir.

### 7.3 Game batch kaydetme

`train.py` artık her oyun için batch kaydedebilir:

```bash
--save-game-batches --batch-save-dir checkpoints/game_batches
```

Her `.pt` dosyasında:

```text
iteration, episode, seed
transition_start, transition_end, num_transitions
episode_stats
train_cfg, ppo_cfg, reward_cfg
batch
```

`batch` alanında:

```text
planet_feats, fleet_feats, global_feats, edge_bias
planet_mask, fleet_mask, source_mask
src, tgt, ship_value, is_noop, player    # ship_value continuous float
old_log_prob, old_value, returns, advantages
```

Saved batch içinde `advantages` ham GAE değeridir; PPO training sırasında `iter_minibatches` bunu buffer genelinde normalize eder.

## 8. Çalıştırma: Komutlar

### 8.1 Önerilen koşum (grouped_planet + production_simple, scripted opponent)

```bash
orbit_wars_venv/bin/python -m rl.train \
  --reward-system production_simple \
  --opponent scripted \
  --device cuda \
  --num-agents 2 \
  --iterations 200 \
  --episodes-per-iter 8 \
  --max-episode-steps 500 \
  --model-variant grouped_planet \
  --d-model 128 --n-heads 4 --n-layers 3 \
  --lr 5e-5 \
  --gamma 0.99 --gae-lambda 0.95 \
  --clip-range 0.2 --ppo-epochs 4 --batch-size 256 \
  --target-kl 0.05 \
  --entropy-coef 0.05 \
  --defense-kills-weight 0.01 \
  --capture-value-weight 0.01 \
  --capture-time-curvature 1.5 \
  --ship-overflow-weight 1.0 \
  --scripted-expansion-turns 20 \
  --checkpoint-every 10 \
  --checkpoint-dir checkpoints/run_v2 \
  --log-path checkpoints/run_v2/train.jsonl \
  --seed 42
```

### 8.2 CPU smoke test

```bash
orbit_wars_venv/bin/python -m rl.train \
  --reward-system production_simple \
  --opponent scripted \
  --device cpu \
  --iterations 5 --episodes-per-iter 2 --max-episode-steps 200 \
  --num-agents 2 --model-variant grouped_planet \
  --d-model 64 --n-heads 2 --n-layers 2 \
  --lr 5e-5 --ppo-epochs 2 --batch-size 128 \
  --target-kl 0.05 --entropy-coef 0.05 \
  --checkpoint-dir /tmp/orbit_smoke \
  --log-path /tmp/orbit_smoke/train.jsonl --seed 0
```

İlk birkaç epoch'ta sağlıklı durumda görmen beklenen değerler:
- `KL` < 0.01
- `clip` < 0.05
- `H` pozitif (3 civarı, p_max ve action space'a göre)
- `αβ` (1.5, 1.5) civarı, ilerledikçe yavaş yükselir
- `r=[q05,q50,q95]` ≈ [0.95, 1.00, 1.05] (dar bant)

### 8.3 Resume

```bash
orbit_wars_venv/bin/python -m rl.train \
  --resume-from checkpoints/run_v2/latest.pt \
  --checkpoint-dir checkpoints/run_v2 \
  --log-path checkpoints/run_v2/train.jsonl \
  --reward-system production_simple --opponent scripted --model-variant grouped_planet \
  --device cuda --iterations 400 --episodes-per-iter 8
```

### 8.4 Replay izleme (watch)

```bash
orbit_wars_venv/bin/python -m rl.watch \
  --ckpt checkpoints/run_v2/latest.pt \
  --html replays/run_v2.html \
  --device cuda \
  --deterministic
```

`--deterministic` watch'ta Beta dağılımının ortalaması ($\alpha/(\alpha+\beta)$) kullanılır. Stokastik
izlemek için bayrağı kaldır.

## 9. Test Matrisi ve Sanity Checks

`tests/test_rl.py` (117+ test) başlıca şunları kanıtlar:

- `RolloutBuffer.add_delayed_reward` `reward * discount**delay` uyguluyor mu?
- `compute_gae` oyuncuların transition'larını ayrı işliyor mu?
- `act_source` ile üretilen log-prob, `evaluate_source_actions` ile tekrar hesaplanınca eşleşiyor mu?
- `production_simple` capture/combat/defense-deficit formülleri beklenen ölçeği veriyor mu?
- `normalize_terminal_rewards(..., draw_value=...)` draw değerini doğru yayıyor mu?

`tests/test_obs_builder.py`:

- `PLANET_FEATURE_DIM = 34`, `GLOBAL_FEATURE_DIM = 14`, `EDGE_FEATURE_DIM = 6`.
- Incoming bucket ve aggregate threat alanları beklenen indekslerde.
- `decode_action_with_launch` Beta sample'ı `ship_value × SHIP_REQUEST_CAP` ile decode ediyor; overflow info'su (`raw_request`, `overflow_ships`) doğru.

`tests/test_model.py`:

- Forward shape'leri (özellikle `ship_params (B, P, P, 2)` grouped variant'ta).
- `act` ve `act_source` source'u sabit tutuyor ve `evaluate_source_actions` log-prob eşleşiyor.
- Beta concentration soft-cap çalışıyor (α, β ≤ 30).

## 10. Kısa Özet

- **Observation**: permutation-invariant set; `PLANET_FEATURE_DIM=34`, `GLOBAL_FEATURE_DIM=14`, `EDGE_FEATURE_DIM=6`; geometrik edge bias; incoming fleet özetleri planet feature içinde.
- **Model**: biased Set Transformer; default `grouped_planet`; planet-only token; **pair-conditioned Beta head** (α, β ∈ [1, 30] soft-cap).
- **Actor heads**: src + noop + bilinear pair pointer + pair-conditioned Beta. NOOP target'larla aynı softmax içinde yarışır (per-source kategorical effectively pair-aware).
- **Rotation**: `is_rotating`, `omega`, `r_sun`, `cos(theta)`, `sin(theta)` ve orbital edge bias modelde var; ama future reachability/opportunity window explicit feature değil.
- **Action decode**: policy açı üretmez; `compute_launch` env için `[from_id, angle_in_radians, ships]` üretir ve spawn offset + collision kontrollerini deterministic çözer.
- **Öğrenme**: clipped PPO + GAE + KL early-stop; PPO update `evaluate_source_actions` ile stored action log-prob'unu tekrar hesaplar. Importance ratio quantile + Beta α/β logging trust-region sağlığını gösterir.
- **Batch**: `RolloutBuffer` transition → GAE → normalized minibatch; isteğe bağlı `--save-game-batches` ile her oyun `.pt` olarak saklanır.

Daha derin okuma için: `docs/end_to_end_walkthrough.ipynb`, `docs/grouped_planet_model_walkthrough.ipynb`, `docs/reward_system_explained.ipynb`.